# TV-00b C# — Variantes d’attention : MHA, MQA, GQA, SWA

Le notebook précédent, [RoPE C#](TV-00a-RoPE-from-scratch-CSharp.ipynb), ouvrait la boîte du **positionnement** ; celui-ci ouvre celle de l’**attention** telle qu’elle est réellement câblée dans LLaMA-2/3, Mistral ou Qwen.

Un modèle comme Mistral-7B n’exécute pas `attention(Q, K, V)` avec 32 têtes identiques. Trois variantes illustrent les compromis étudiés ici. Mistral-7B v0.1 combine GQA et SWA :

1. **MQA** (*Multi-Query Attention*) : une seule paire de projections K/V pour toutes les têtes Q. Le cache KV de décodage s’effondre d’un facteur égal au nombre de têtes.
2. **GQA** (*Grouped-Query Attention*) : les têtes Q sont réparties en groupes qui **partagent** une tête K/V. C’est le réglage de LLaMA-2 70B (64 têtes Q, 8 têtes KV) et de Mistral, un compromis entre la qualité de MHA et la mémoire de MQA.
3. **SWA** (*Sliding Window Attention*) : chaque requête ne lit que les `W` dernières clés. À largeur fixe, une implémentation en bande rend le nombre de scores linéaire en longueur. La portée directe de chaque requête se limite à la fenêtre.

L’objectif est de **coder** ces variantes, puis de **mesurer** leur coût et leurs apports sur un banc dont chaque paramètre est contrôlé.

| Question | Instrument |
|---|---|
| Les variantes sont-elles correctes ? | Échelle de vérifications exactes : boucle par tête, invariants de groupement, identité de masque |
| Combien de mémoire le cache KV économise-t-il ? | Formule analytique confrontée à une allocation réelle |
| Le partage des têtes KV accélère-t-il l’entraînement ? | Chronométrage du passage avant et arrière |
| La fenêtre glissante accélère-t-elle vraiment ? | Bande explicite `O(T*W)` contre masque `O(T^2)`, avec balayage de `T` |
| Que coûte la fenêtre en qualité ? | Tâche de rappel à longue portée, où la distance dépasse la fenêtre |

Tout tourne sur CPU avec TorchSharp après restauration du package NuGet : aucun modèle externe ni `transformers`, `xformers` ou `flash-attn`.

[Version Python](TV-00b-Attention-Variants-from-scratch.ipynb) | [RoPE C#](TV-00a-RoPE-from-scratch-CSharp.ipynb)

## Sommaire

1. L’attention causale multi-têtes et sa brique de référence
2. Le cache KV à l’inférence
3. GQA et MQA, le partage des têtes KV
4. SWA et la bande locale
5. Temps de calcul
6. Rappel d’un marqueur distant
7. Conclusion et suite


In [1]:
#r "nuget:TorchSharp-cpu, 0.107.0"
// Le même thread et la même précision servent aux mesures avant et arrière.
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using TorchSharp;
using static TorchSharp.torch;
using static TorchSharp.torch.nn;
using Tensor = TorchSharp.torch.Tensor;

torch.set_num_threads(1);
torch.manual_seed(0);
Console.WriteLine($"TorchSharp {typeof(torch).Assembly.GetName().Version}, libtorch {torch.__version__}, CPU, threads {torch.get_num_threads()}");


Installed Packages TorchSharp-cpu, 0.107.0

TorchSharp 0.107.0.0, libtorch 2.10.0.0, CPU, threads 1


## 1. L'attention causale multi-tetes

Rappel de la forme canonique, tete `h` comprise :

```
scores[h] = Q[h] K[h]^T / sqrt(d_head)
attn[h]   = softmax(mask_causal(scores[h]))
sortie[h] = attn[h] V[h]
```

avec `Q = x W_q`, `K = x W_k`, `V = x W_v`, une concatenation des tetes en sortie, puis `W_o`.
Le masque causal est triangulaire : la requête `i` ne lit que les clés `j <= i`. C'est ce masque qui
rend le modèle autoregressif, et c'est lui aussi qui rend le **cache KV** possible : a la generation,
la cle et la valeur de chaque position déjà produite ne changent plus, on les garde.

On code d'abord une brique de reference **non vectorisee** — une boucle explicite sur les tetes. Elle
est lente et c'est voulu : elle sert de verite contre laquelle verifier la forme vectorisee que tout
le reste du notebook utilise. Une implementation d'attention qui n'est jamais confrontee a une
seconde implementation independante est une implementation dont on ne sait rien.

In [2]:
// La variante fixe le nombre de têtes KV à la construction. Les masques restent des opérations de tenseurs différentiables.
public class Variant : Module<Tensor, Tensor>
{
    readonly long h, g, dh;
    readonly int? window;
    readonly bool banded;
    readonly Module<Tensor, Tensor> q, k, v, o;
    public Variant(int d, int heads, int groups, int? w = null, bool band = false) : base("Variant")
    {
        h = heads;
        g = groups;
        dh = d / heads;
        window = w;
        banded = band;
        q = torch.nn.Linear(d, heads * dh, hasBias: false);
        k = torch.nn.Linear(d, groups * dh, hasBias: false);
        v = torch.nn.Linear(d, groups * dh, hasBias: false);
        o = torch.nn.Linear(d, d, hasBias: false);
        RegisterComponents();
    }

    // Le masque causal interdit les positions futures avant la normalisation.
    public static Tensor Mask(Tensor q, Tensor k, Tensor v, int? w)
    {
        var t = q.shape[2];
        var i = arange(t).reshape(t, 1);
        var j = arange(t).reshape(1, t);
        var mask = j.le(i);
        if (w.HasValue)
            mask = mask.logical_and((i - j).lt(w.Value));
        var scores = matmul(q, k.transpose(-2, -1)) / Math.Sqrt(q.shape[3]);
        return matmul(scores.masked_fill(mask.logical_not(), double.NegativeInfinity).softmax(-1), v);
    }

    // La bande alloue une fenêtre bornée par requête à la place du carré des positions.
    public static Tensor Band(Tensor q, Tensor k, Tensor v, int window)
    {
        var t = q.shape[2];
        var w = Math.Min(window, (int)t);
        var pad = zeros(new long[] { q.shape[0], q.shape[1], w - 1, q.shape[3] });
        // unfold expose les fenêtres chevauchantes en réutilisant les mêmes valeurs K/V.
        var kw = cat(new[] { pad, k }, 2).unfold(2, w, 1);
        var vw = cat(new[] { pad, v }, 2).unfold(2, w, 1);
        var scores = einsum("bhtd,bhtdw->bhtw", q, kw) / Math.Sqrt(q.shape[3]);
        // La bande exclut les cases factices introduites par le padding gauche.
        var keep = arange(w).reshape(1, 1, 1, w).ge((w - 1 - arange(t)).reshape(1, 1, t, 1));
        return einsum("bhtw,bhtdw->bhtd", scores.masked_fill(keep.logical_not(), double.NegativeInfinity).softmax(-1), vw);
    }

    public long KvParams() => k.parameters().Sum(p => p.numel()) + v.parameters().Sum(p => p.numel());
    public override Tensor forward(Tensor x)
    {
        var b = x.shape[0];
        var t = x.shape[1];
        var Q = q.forward(x).reshape(b, t, h, dh).transpose(1, 2);
        var K = k.forward(x).reshape(b, t, g, dh).transpose(1, 2);
        var V = v.forward(x).reshape(b, t, g, dh).transpose(1, 2);
        // Le groupe KV est partagé entre les têtes Q qui lui correspondent.
        if (g != h)
        {
            K = K.repeat_interleave(h / g, 1);
            V = V.repeat_interleave(h / g, 1);
        }

        var z = banded && window.HasValue ? Band(Q, K, V, window.Value) : Mask(Q, K, V, window);
        return o.forward(z.transpose(1, 2).reshape(b, t, h * dh));
    }
}

Tensor Naive(Tensor q, Tensor k, Tensor v)
{
    var heads = new List<Tensor>();
    for (int h = 0; h < q.shape[1]; h++)
    {
        var Q = q.select(1, h);
        var K = k.select(1, h);
        var V = v.select(1, h);
        var t = q.shape[2];
        var mask = torch.triu(torch.ones(new long[] { t, t }, dtype: ScalarType.Bool), 1);
        var s = torch.matmul(Q, K.transpose(-2, -1)) / Math.Sqrt(q.shape[3]);
        heads.Add(torch.matmul(s.masked_fill(mask, double.NegativeInfinity).softmax(-1), V));
    }

    return torch.stack(heads.ToArray(), 1);
}

Console.WriteLine("Attention causale par tête, masque vectorisé et bande définis.");

Attention causale par tête, masque vectorisé et bande définis.


### 1.1 Echelle de verifications

Avant de mesurer quoi que ce soit, on etablit que les briques sont ce qu'elles pretendent etre. Cinq
verifications, chacune confrontant le code a une **propriete enoncee independamment** :

- **(a)** la forme vectorisee (reshape + broadcasting) reproduit la boucle tete par tete ;
- **(b)** une fenêtre au moins aussi grande que la sequence ne mord pas : le masque fenêtre est alors
  le masque causal pur, exactement ;
- **(c)** le nombre de paires (requête, cle) lues sous fenêtre `W` vaut exactement
  `T*W - W(W-1)/2` des que `T >= W` — la quantite qui fait passer le cout de `O(T^2)` a `O(T*W)` ;
- **(d)** l'etagement des tetes KV (section 3) place la tete Q `h` sur la tete KV `h // (H/G)` ;
- **(e)** la bande explicite de la section 4 redonne le résultat du masque, au bruit flottant pres.

Aucune de ces verifications n'utilise le résultat qu'elle valide. C'est le point : `(a)` et `(e)`
comparent deux chemins de calcul independants, `(b)` et `(c)` comparent le code a une formule
fermee, `(d)` a un invariant de groupement.

In [3]:
// La boucle par tête sert de référence indépendante au calcul vectorisé sur les mêmes entrées.
var inputs = torch.randn(new long[] { 2, 16, 64 });
// Les projections de départ sont les mêmes pour le contrôle vectorisé et la boucle indépendante.
var weightGen = new torch.Generator(3, torch.CPU);
var wq = torch.randn(new long[] { 64, 64 }, generator: weightGen) / Math.Sqrt(64);
var wk = torch.randn(new long[] { 64, 64 }, generator: weightGen) / Math.Sqrt(64);
var wv = torch.randn(new long[] { 64, 64 }, generator: weightGen) / Math.Sqrt(64);
var wo = torch.randn(new long[] { 64, 64 }, generator: weightGen) / Math.Sqrt(64);
var q = torch.matmul(inputs, wq).reshape(2, 16, 4, 16).transpose(1, 2);
var k = torch.matmul(inputs, wk).reshape(2, 16, 4, 16).transpose(1, 2);
var v = torch.matmul(inputs, wv).reshape(2, 16, 4, 16).transpose(1, 2);
var refH = torch.matmul(Naive(q, k, v).transpose(1, 2).reshape(2, 16, 64), wo);
var vec = torch.matmul(Variant.Mask(q, k, v, null).transpose(1, 2).reshape(2, 16, 64), wo);
var diff = (refH - vec).abs().max().item<float>();
Console.WriteLine($"(a) boucle / vectorisé : {diff:E3}");
if (diff > 1e-5)
    throw new Exception("attention par tête incorrecte");
foreach (var(T, W)in new[]
{
    (64, 64),
    (64, 200)
}

)
{
    var i = torch.arange(T).reshape(T, 1);
    var j = torch.arange(T).reshape(1, T);
    var a = j.le(i);
    var b = a.logical_and((i - j).lt(W));
    Console.WriteLine($"(b) T={T}, W={W}, différence masque : {(a.to_type(ScalarType.Int32) - b.to_type(ScalarType.Int32)).abs().sum().item<long>()}");
}

foreach (var(T, W)in new[]
{
    (64, 16),
    (64, 32),
    (100, 7)
}

)
{
    var i = torch.arange(T).reshape(T, 1);
    var j = torch.arange(T).reshape(1, T);
    var n = j.le(i).logical_and((i - j).lt(W)).sum().item<long>();
    var formula = T * W - W * (W - 1) / 2;
    Console.WriteLine($"(c) T={T}, W={W} : {n}, formule {formula}");
    if (n != formula)
        throw new Exception("paires");
}

(a) boucle / vectorisé : 0.000E+000


(b) T=64, W=64, différence masque : 0


(b) T=64, W=200, différence masque : 0


(c) T=64, W=16 : 904, formule 904


(c) T=64, W=32 : 1552, formule 1552


(c) T=100, W=7 : 679, formule 679


### 1.2 Lecture

Les contrôles comparent la boucle par tête et le calcul vectorisé, puis vérifient les masques et le comptage des paires. Les écarts doivent rester sous la tolérance.

## 2. Le cache KV à l’inférence

Dans le calcul dense, MHA, GQA et MQA produisent une grille de scores de même taille pour les têtes Q. Le partage réduit les projections K et V et la taille du cache. Pour SWA, le nombre de scores effectivement calculés dépend du choix entre masque dense et bande, étudié au § 4.

Lors de la génération autorégressive, un nouveau jeton ajoute une clé et une valeur par tête KV et par couche. Avec un historique complet, la taille du cache vaut

```
octets = 2 (K et V) × B × têtes KV × T × d_head × taille élément × couches
```

Un élément FP16 occupe deux octets. On applique la formule à des architectures réelles, puis on alloue un tenseur FP16 et on double sa taille pour représenter le couple K et V. La longueur retenue pour ce calcul est commune aux architectures. Une implémentation SWA qui évince les clés et valeurs hors fenêtre peut aussi borner la longueur mise en cache.


In [4]:
// Le cache stocke une clé et une valeur par jeton, par tête KV et par couche.
long Cache(long layers, long groups, long dh, long T, long batch = 1, long octets = 2) => 2 * batch * layers * groups * T * dh * octets;
foreach (var(name, d, h, g, l)in new[]
{
    ("LLaMA-2 7B", 4096, 32, 32, 32),
    ("LLaMA-2 70B", 8192, 64, 8, 80),
    ("Mistral 7B", 4096, 32, 8, 32),
    ("notre jouet", 64, 4, 4, 2)
}

)
{
    var mha = Cache(l, h, d / h, 32768);
    var kv = Cache(l, g, d / h, 32768);
    Console.WriteLine($"{name} Q={h} KV={g} MHA={mha / (double)(1L << 30):F2} Gio livré={kv / (double)(1L << 30):F2} Gio rapport={mha / (double)kv:F1}");
}

// La mesure FP16 tient compte des deux octets par entrée et des deux tenseurs.
var cache = torch.zeros(new long[] { 1, 8, 4096, 128 }, dtype: ScalarType.Float16);
long measured = 2 * cache.numel() * 2;
Console.WriteLine($"Cache FP16 K/V : mesuré {measured}, formule {Cache(1, 8, 128, 4096)} octets");
if (measured != Cache(1, 8, 128, 4096))
    throw new Exception("cache");

LLaMA-2 7B Q=32 KV=32 MHA=16.00 Gio livré=16.00 Gio rapport=1.0


LLaMA-2 70B Q=64 KV=8 MHA=80.00 Gio livré=10.00 Gio rapport=8.0


Mistral 7B Q=32 KV=8 MHA=16.00 Gio livré=4.00 Gio rapport=4.0


notre jouet Q=4 KV=4 MHA=0.02 Gio livré=0.02 Gio rapport=1.0


Cache FP16 K/V : mesuré 16777216, formule 16777216 octets


### 2.1 Lecture

Le cache comporte deux tenseurs K et V. Le tableau utilise FP16 et le même contexte ; le contrôle final compare la formule au double de la taille d’un tenseur FP16.

Les lignes de configuration utilisent toutes le même historique hypothétique de 32 768 jetons pour isoler le facteur de partage. Ces capacités calculées décrivent les seuls tenseurs de clés et de valeurs. Les poids, activations temporaires et métadonnées ne figurent pas dans ce calcul. Le contrôle de petite taille affiche des octets, tandis que le tableau des architectures convertit en gibioctets avec une division par 2³⁰.


## 3. GQA et MQA, le partage des têtes KV

La classe `Variant` prend le nombre de groupes et couvre les trois régimes.

| Têtes KV | Nom | Ce que lit la tête Q `h` |
|---|---|---|
| Autant que de têtes Q | MHA | Sa propre paire K/V |
| 1 | MQA | L’unique paire K/V |
| `g` intermédiaire | GQA | La paire K/V du groupe `h / rep`, en division entière |

Ici, `rep` est le nombre de têtes Q par groupe. L’appel TorchSharp `repeat_interleave(rep, 1)` répète les **activations** K et V pour former les tenseurs consommés par le calcul dense. Chaque projection conserve ses propres poids, dont le nombre est proportionnel aux têtes KV. Les poids K et V ne sont pas dupliqués par l’opération. La comparaison de qualité oppose donc des variantes dont les nombres de paramètres diffèrent.


In [5]:
// Le rapport de stockage mesure les tenseurs de projection K et V réellement alloués.
var kv0 = torch.randn(new long[] { 1, 2, 5, 4 });
// Quatre têtes Q consécutives lisent la même tête KV dans ce groupe.
var expanded = kv0.repeat_interleave(4, 1);
Console.WriteLine($"(d) forme : {string.Join(",", expanded.shape)}");
for (int h = 0; h < 8; h++)
{
    var d = (expanded.select(1, h) - kv0.select(1, h / 4)).abs().max().item<float>();
    if (d != 0)
        throw new Exception("groupement");
}

Console.WriteLine("Chaque tête Q lit la tête KV attendue.");
foreach (int g in new[]
{
    32,
    8,
    1
}

)
{
    using var scope = torch.NewDisposeScope();
    using var m = new Variant(4096, 32, g);
    long expected = 2L * 4096 * g * 128;
    var actual = m.KvParams();
    if (actual != expected)
        throw new Exception("compte K/V");
    Console.WriteLine($"groupes {g} : K/V {actual:N0} paramètres ; total module {m.parameters().Sum(p => p.numel()):N0}");
}

(d) forme : 1,8,5,4


Chaque tête Q lit la tête KV attendue.


groupes 32 : K/V 33,554,432 paramètres ; total module 67,108,864


groupes 8 : K/V 8,388,608 paramètres ; total module 41,943,040


groupes 1 : K/V 1,048,576 paramètres ; total module 34,603,008


### 3.1 Lecture

Le partage répète chaque tête KV pour un groupe de têtes Q. Les paramètres K/V diminuent avec le nombre de groupes ; les autres projections subsistent.

Pour refaire le calcul à la main, une projection K envoie chaque vecteur d’entrée vers un vecteur de taille `têtes KV × d_head` et V utilise une projection distincte de même taille. Dans cet exemple sans biais, les deux projections totalisent donc `2 × d_model × têtes KV × d_head` poids. La projection Q conserve une composante par tête de requête, et la projection de sortie conserve la dimension du modèle.


## 4. SWA : la bande locale

L'attention a fenêtre glissante restreint chaque requête aux `W` dernières clés. Le masque est
l'intersection de deux conditions : causale (`j <= i`) et locale (`i - j < W`). Le comptage de la
verification `(c)` dit déjà l'essentiel : `T*w - w(w-1)/2` paires au lieu de `T(T+1)/2`, avec `w = min(W,T)`.

Il y a cependant **deux facons de coder** cette restriction, et la différence entre les deux est le
sujet d'une bonne partie de la section 5 :

- **par masque** : on calcule les `T x T` scores comme d'habitude, puis on remplace par `-inf` tout ce
  qui sort de la bande. C'est direct, lisible, et le cout reste `O(T^2)` : on a calcule les scores
  hors bande pour les jeter. Le calcul reste dense. L’économie du cache de décodage dépend séparément de la politique de conservation des K/V.
- **par bande** : on ne materialise que les `W` colonnes utiles en decalant les clés, ce qui donne un
  tenseur `(B, H, T, W)`. Le cout devient reellement `O(T*W)`, mais plus aucune opération n'est un
  produit matriciel dense — et c'est précisément la le piege que la section 5 va mesurer.

Les deux doivent donner le même résultat. C'est la verification `(e)` : elle autorise a utiliser la
bande pour les mesures de vitesse sans avoir a croire sa correction sur parole.

In [6]:
// Chaque paire T et W compare la même attention et vérifie également son gradient.
foreach (var(T, W)in new[]
{
    (64, 16),
    (64, 33),
    (128, 5),
    (37, 37)
}

)
{
    using var scope = torch.NewDisposeScope();
    var a = torch.randn(new long[] { 2, 4, T, 8 });
    var b = torch.randn(new long[] { 2, 4, T, 8 });
    var c = torch.randn(new long[] { 2, 4, T, 8 });
    var x = Variant.Mask(a, b, c, W);
    var y = Variant.Band(a, b, c, W);
    var err = (x - y).abs().max().item<float>();
    Console.WriteLine($"T={T}, W={W}, masque/bande {err:E3}");
    if (err > 1e-5)
        throw new Exception("bande incorrecte");
}

torch.manual_seed(0);
// Le contrôle en gradient compare des modules appliqués aux mêmes entrées.
var m1 = new Variant(64, 4, 4, 16, false);
var m2 = new Variant(64, 4, 4, 16, true);
m2.load_state_dict(m1.state_dict());
var x1 = torch.randn(new long[] { 2, 32, 64 }, requires_grad: true);
var x2 = x1.detach().clone().requires_grad_(true);
var y1 = m1.forward(x1);
var y2 = m2.forward(x2);
var forward = (y1 - y2).abs().max().item<float>();
y1.sum().backward();
y2.sum().backward();
var grad = (x1.grad - x2.grad).abs().max().item<float>();
Console.WriteLine($"même module : avant {forward:E3}, gradient {grad:E3}");
if (forward > 1e-5 || grad > 1e-5)
    throw new Exception("écart module/gradient");

T=64, W=16, masque/bande 3.576E-007


T=64, W=33, masque/bande 4.768E-007


T=128, W=5, masque/bande 3.576E-007


T=37, W=37, masque/bande 4.768E-007


même module : avant 1.192E-007, gradient 3.576E-007


### 4.1 Lecture

La bande et le masque sont confrontés sur plusieurs tailles, puis au niveau du module avec les mêmes poids et gradients.

## Exercice 1 — le groupement des têtes Q

Compléter `BuildKvHeadsParVue(kv, heads, groups)` avec des opérations de vue et de diffusion, puis vérifier ses valeurs contre la référence `repeat_interleave`. Le résultat doit avoir la forme `(B, heads, T, dh)`.

**Indice.** Faites apparaître un axe de taille 1 entre le groupe et la longueur, diffusez-le à `heads / groups`, puis regroupez les deux axes des têtes.

Le test de mutation de la cellule distingue l’égalité des valeurs du partage du stockage. Une vue diffusée peut partager le stockage alors que le regroupement par `reshape` matérialise des valeurs. Décrivez le résultat observé avec deux têtes KV.


In [7]:
Tensor BuildKvHeadsParVue(Tensor kv, int heads, int groups)
{ // TODO : view, expand puis reshape ; vérifier le stockage et les valeurs.
    return null;
}

var kvEssai = torch.randn(new long[] { 2, 2, 6, 4 });
var attendu = kvEssai.repeat_interleave(4, 1);
var obtenu = BuildKvHeadsParVue(kvEssai, 8, 2);
if (obtenu is null)
    Console.WriteLine("Exercice 1 à compléter : résultat null");
else
{
    Console.WriteLine($"forme {string.Join(",", obtenu.shape)}, écart {(obtenu - attendu).abs().max().item<float>()}");
    var before = obtenu.clone();
    kvEssai.add_(1);
    var share = (obtenu - before).abs().max().item<float>() > 0;
    kvEssai.sub_(1);
    Console.WriteLine($"stockage partagé (test de mutation) : {share}");
}


Exercice 1 à compléter : résultat null


## 5. Temps de calcul des variantes

Le premier banc chronomètre le passage avant et arrière d’une couche d’attention à deux longueurs. Le second confronte le masque et la bande sur plusieurs longueurs de séquence.

Les mesures utilisent TorchSharp sur un seul thread CPU. Elles portent sur l’entraînement d’une séquence complète. En génération autorégressive avec quatre ou huit fois moins de têtes KV, le volume théorique des clés et valeurs à lire diminue du même facteur à historique égal. La latence dépend aussi du noyau, de la bande passante et des autres calculs.

Les variantes sont chronométrées en tours entrelacés pour répartir l’effet de l’ordre et des allocations. Le contrôle avec `W ≥ T` vérifie que les variantes comparées réalisent le même calcul utile.


In [8]:
// Les essais sont entrelacés pour que la variante ne soit pas toujours mesurée la première.
Dictionary<string, double> Bench(List<(string, Variant)> modules, int T, int batch = 2, int warm = 3, int rounds = 9)
{
    using var x = torch.randn(new long[] { batch, T, 128 }, requires_grad: true);
    foreach (var(_, m)in modules)
        for (int i = 0; i < warm; i++)
        {
            using var scope = torch.NewDisposeScope();
            m.forward(x).sum().backward();
        }

    // Après chauffe, garder la médiane des mesures de chaque variante.
    var samples = modules.ToDictionary(p => p.Item1, p => new List<double>());
    for (int i = 0; i < rounds; i++)
        foreach (var(name, m)in modules)
        {
            using var scope = torch.NewDisposeScope();
            m.zero_grad();
            var sw = Stopwatch.StartNew();
            m.forward(x).sum().backward();
            sw.Stop();
            samples[name].Add(sw.Elapsed.TotalMilliseconds);
        }

    return samples.ToDictionary(p => p.Key, p => p.Value.OrderBy(x => x).ElementAt(rounds / 2));
}

Variant Make(int groups, int? window = null, bool band = false)
{
    torch.manual_seed(0);
    return new Variant(128, 4, groups, window, band);
}

foreach (int T in new[]
{
    128,
    512
}

)
{
    var mods = new List<(string, Variant)>
    {
        ("MHA", Make(4)),
        ("GQA", Make(2)),
        ("MQA", Make(1)),
        ("SWA", Make(4, 128))
    };
    var t = Bench(mods, T);
    Console.WriteLine($"T={T}: " + string.Join(" | ", t.Select(x => $"{x.Key} {x.Value:F2} ms")));
    Console.WriteLine($"SWA/MHA {t["SWA"] / t["MHA"]:F2}, GQA/MHA {t["GQA"] / t["MHA"]:F2}, MQA/MHA {t["MQA"] / t["MHA"]:F2}");
    foreach (var(_, m)in mods)
    {
        Console.WriteLine($"paramètres {m.parameters().Sum(p => p.numel())}");
        m.Dispose();
    }
}

T=128: MHA 3.92 ms | GQA 4.13 ms | MQA 4.08 ms | SWA 5.41 ms


SWA/MHA 1.38, GQA/MHA 1.06, MQA/MHA 1.04


paramètres 65536


paramètres 49152


paramètres 40960


paramètres 65536


... (2 outputs left out)

paramètres 65536


paramètres 49152


paramètres 40960


paramètres 65536


La mesure suivante prolonge le banc précédent avec les mêmes conditions.

In [9]:
// À W fixé, le triangle causal croît plus vite que le nombre de paires de fenêtre.
foreach (int T in new[]
{
    128,
    512,
    1024
}

)
{
    var mods = new List<(string, Variant)>
    {
        ("masque", Make(4, 128)),
        ("bande", Make(4, 128, true))
    };
    var t = Bench(mods, T, batch: 1);
    var p = T * (T + 1) / 2;
    // Les premières requêtes ne remplissent pas encore leur fenêtre.
    var band = Enumerable.Range(1, T).Sum(i => Math.Min(i, 128));
    Console.WriteLine($"T={T} masque {t["masque"]:F2} ms, bande {t["bande"]:F2} ms, rapport {t["masque"] / t["bande"]:F2}, paires {p}/{band}");
    foreach (var(_, m)in mods)
        m.Dispose();
}

T=128 masque 2.78 ms, bande 25.60 ms, rapport 0.11, paires 8256/8256


T=512 masque 24.69 ms, bande 182.97 ms, rapport 0.13, paires 131328/57408


T=1024 masque 191.71 ms, bande 423.89 ms, rapport 0.45, paires 524800/122944


### 5.1 Lecture

Les médianes sont mesurées en tours entrelacés, trois chauffes et neuf répétitions. Les rapports sont propres à ce replay CPU et ne prouvent pas un gain universel. Le contrôle W ≥ T compare la même computation.

Les comptes de paramètres après chaque tableau suivent le même ordre que les mesures : MHA, GQA, MQA, puis SWA. Le contrôle W ≥ T compare deux calculs identiques mais leurs temps peuvent différer sous contention. La bande explicite évite des paires, ce qui ne suffit pas à garantir une latence inférieure : allocation, accès mémoire et noyaux utilisés comptent aussi.

Le tableau suivant permet de relier le chronomètre aux paires autorisées. Lorsque la fenêtre couvre toute la séquence, toutes les paires causales sont présentes. Pour une séquence plus longue, le rapport de paires décrit le travail mathématique utile. La mesure inclut aussi la préparation des entrées de la bande, les allocations et leur coût dans la passe arrière. Pour interpréter un rapport `masque / bande`, une valeur supérieure à 1 signifie que la bande a été plus rapide dans ces conditions.


## Exercice 2 — le masque en bande

Compléter `MasqueBande(T, W)` pour obtenir les positions valides sous forme d’un tenseur booléen `(T, W)`, diffusable en `(1, 1, T, W)`. Construire le masque sans boucle sur `T` et en partant des indices absolus des clés.

**Indice.** À l’emplacement `d` de la bande pour la requête `i`, la clé correspond à la position absolue `i + d - (W - 1)`. Deux `torch.arange` et des axes de longueur 1 permettent de combiner les indices par diffusion.

**Contrôle.** Comparer à la bande utilisée par `Variant.Band`. La ligne `i` contient `min(i + 1, W)` positions vraies. Pour `w = min(W, T)`, le total vaut `T*w - w(w-1)/2`.


In [10]:
Tensor MasqueBande(int T, int W)
{ // TODO : deux arange et une comparaison diffusée
    return null;
}

var mb = MasqueBande(64, 16);
if (mb is null)
    Console.WriteLine("Exercice 2 à compléter : résultat null");
else
{
    var expectedCounts = torch.tensor(Enumerable.Range(0, 64).Select(i => (long)Math.Min(i + 1, 16)).ToArray());
    Console.WriteLine($"forme {string.Join(",", mb.shape)}, écart comptes {(mb.sum(1) - expectedCounts).abs().max().item<long>()}, total {mb.sum().item<long>()}, formule {64 * 16 - 16 * 15 / 2}");
}


Exercice 2 à compléter : résultat null



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



## 6. Rappel d’un marqueur distant

La séquence présente d’abord un marqueur tiré parmi huit, puis des jetons de remplissage, puis une requête. À la dernière position, le modèle doit retrouver le marqueur situé à distance 63. Le hasard correspond à une exactitude de 1/8 et la distribution uniforme à une perplexité de 8.

Une couche SWA voit les `W` positions de sa fenêtre, dont la position courante. Une pile de `L` couches couvre au plus `L * (W - 1) + 1` positions. Les modèles de la comparaison principale ont deux couches. Pour `W ≤ 32`, ils peuvent remonter de 62 positions au plus et le marqueur est hors de portée par construction.

La fenêtre complète `W = T = 64` contrôle le chemin SWA. MHA, GQA et MQA fournissent trois autres variantes de portée complète. Le diagnostic de profondeur fixe `W = 16` : il faut au moins cinq couches pour que le champ théorique couvre le marqueur. Les mesures de profondeur indiquent si les modèles l’apprennent dans le budget donné.

Le tableau évalue l’exactitude et la perplexité à la position de requête. Les jetons de remplissage sont aléatoires et ne font pas partie de cette mesure.


In [11]:
// Dans la tâche de rappel, le marqueur se trouve au début et seul le dernier logit supervise sa lecture.
public class Bloc : Module<Tensor, Tensor>
{
    readonly Module<Tensor, Tensor> ln1, ln2, up, down, act;
    readonly Variant attn;
    public Bloc(int d, int h, int g, int? w) : base("Bloc")
    {
        ln1 = torch.nn.LayerNorm(new long[] { d });
        ln2 = torch.nn.LayerNorm(new long[] { d });
        attn = new Variant(d, h, g, w);
        up = torch.nn.Linear(d, 4 * d);
        down = torch.nn.Linear(4 * d, d);
        act = torch.nn.GELU();
        RegisterComponents();
    }

    public override Tensor forward(Tensor x)
    {
        var y = x + attn.forward(ln1.forward(x));
        return y + down.forward(act.forward(up.forward(ln2.forward(y))));
    }
}

public class Petit : Module<Tensor, Tensor>
{
    readonly Module<Tensor, Tensor> emb, ln, head;
    readonly Bloc b0, b1, b2, b3, b4, b5;
    readonly int layers;
    public Petit(int vocab, int d, int h, int g, int? w, int L) : base("Petit")
    {
        layers = L;
        emb = torch.nn.Embedding(vocab, d);
        ln = torch.nn.LayerNorm(new long[] { d });
        head = torch.nn.Linear(d, vocab, hasBias: false);
        b0 = new Bloc(d, h, g, w);
        if (L > 1)
            b1 = new Bloc(d, h, g, w);
        if (L > 2)
            b2 = new Bloc(d, h, g, w);
        if (L > 3)
            b3 = new Bloc(d, h, g, w);
        if (L > 4)
            b4 = new Bloc(d, h, g, w);
        if (L > 5)
            b5 = new Bloc(d, h, g, w);
        RegisterComponents();
    }

    public override Tensor forward(Tensor x)
    {
        var y = emb.forward(x);
        y = b0.forward(y);
        if (layers > 1)
            y = b1.forward(y);
        if (layers > 2)
            y = b2.forward(y);
        if (layers > 3)
            y = b3.forward(y);
        if (layers > 4)
            y = b4.forward(y);
        if (layers > 5)
            y = b5.forward(y);
        return head.forward(ln.forward(y));
    }
}

const int VOCAB = 19;
const int T = 64;
// Le dernier jeton est une requête, tandis que la cible supervisée est le premier marqueur.
Tensor Lot(int n, torch.Generator gen)
{
    var marker = torch.randint(0, 8, new long[] { n, 1 }, generator: gen);
    var fill = torch.randint(8, 18, new long[] { n, T - 2 }, generator: gen);
    var query = torch.full(new long[] { n, 1 }, 18, dtype: ScalarType.Int64);
    return torch.cat(new[] { marker, fill, query }, 1);
}

(double, double, double, long) Entrainer(int? window, int layers, int seed, int steps = 200, int groups = 4)
{
    torch.manual_seed(seed);
    using var model = new Petit(VOCAB, 64, 4, groups, window, layers);
    using var opt = torch.optim.Adam(model.parameters(), lr: 0.003);
    var gen = new torch.Generator((ulong)(1234 + seed), torch.CPU);
    var sw = Stopwatch.StartNew();
    for (int i = 0; i < steps; i++)
    {
        using var scope = torch.NewDisposeScope();
        var x = Lot(32, gen);
        // Entraîner seulement le logit final pour mesurer l’accès à distance.
        var logits = model.forward(x).select(1, T - 1);
        var loss = torch.nn.functional.cross_entropy(logits, x.select(1, 0));
        opt.zero_grad();
        loss.backward();
        opt.step();
    }

    sw.Stop();
    using var evalScope = torch.NewDisposeScope();
    using var ng = torch.no_grad();
    // La graine de test indépendante fige des séquences qui ne servent pas à Adam.
    var test = Lot(512, new torch.Generator(99, torch.CPU));
    var output = model.forward(test).select(1, T - 1);
    var lossTest = torch.nn.functional.cross_entropy(output, test.select(1, 0)).item<float>();
    var acc = output.argmax(-1).eq(test.select(1, 0)).to_type(ScalarType.Float32).mean().item<float>();
    return (acc, Math.Exp(lossTest), sw.Elapsed.TotalSeconds, model.parameters().Sum(p => p.numel()));
}

Console.WriteLine("Rappel à distance 63, hasard 1/8, perplexité 8 ; 200 pas, graines 0 et 1.");
foreach (var(name, g, w)in new (string, int, int? )[]
{
    ("MHA", 4, null),
    ("GQA", 2, null),
    ("MQA", 1, null),
    ("SWA W8", 4, 8),
    ("SWA W16", 4, 16),
    ("SWA W32", 4, 32),
    ("SWA W64", 4, 64)
}

)
{
    var a = Entrainer(w, 2, 0, groups: g);
    var b = Entrainer(w, 2, 1, groups: g);
    Console.WriteLine($"{name} accuracy {(a.Item1 + b.Item1) / 2:F3} ± {Math.Abs(a.Item1 - b.Item1) / 2:F3}, ppl {(a.Item2 + b.Item2) / 2:F2}, secondes {b.Item3:F1}, paramètres {b.Item4}");
}

Rappel à distance 63, hasard 1/8, perplexité 8 ; 200 pas, graines 0 et 1.


MHA accuracy 1.000 ± 0.000, ppl 1.00, secondes 12.6, paramètres 102016


GQA accuracy 1.000 ± 0.000, ppl 1.00, secondes 9.3, paramètres 93824


MQA accuracy 1.000 ± 0.000, ppl 1.00, secondes 11.2, paramètres 89728


SWA W8 accuracy 0.126 ± 0.001, ppl 8.09, secondes 10.9, paramètres 102016


SWA W16 accuracy 0.126 ± 0.001, ppl 8.08, secondes 13.0, paramètres 102016


SWA W32 accuracy 0.126 ± 0.001, ppl 8.08, secondes 16.3, paramètres 102016


SWA W64 accuracy 1.000 ± 0.000, ppl 1.00, secondes 16.1, paramètres 102016


La mesure suivante prolonge le banc précédent avec les mêmes conditions.

In [12]:
// La portée théorique croît avec la profondeur, et le test mesure aussi la capacité apprise.
Console.WriteLine("Champ réceptif : L*(W-1)+1 ; distance 63 ; W=16.");
foreach (int L in new[]
{
    1,
    2,
    4,
    6
}

)
{
    var a = Entrainer(16, L, 0, steps: 400);
    Console.WriteLine($"L={L}, champ {L * 15 + 1}, accuracy {a.Item1:F3}, ppl {a.Item2:F2}, secondes {a.Item3:F1}");
}

Champ réceptif : L*(W-1)+1 ; distance 63 ; W=16.


L=1, champ 16, accuracy 0.141, ppl 8.11, secondes 14.0


L=2, champ 31, accuracy 0.141, ppl 8.12, secondes 35.4


L=4, champ 61, accuracy 0.141, ppl 8.12, secondes 82.9


L=6, champ 91, accuracy 0.141, ppl 8.13, secondes 75.6


### 6.1 Lecture

Le rappel exige de transmettre le marqueur initial jusqu’à la requête finale. Comparez les fenêtres, la causalité complète et les champs réceptifs mesurés ci-dessus ; deux graines pour les variantes et une graine pour la profondeur ne suffisent pas à établir une loi générale.

Sur ce replay, MHA, GQA, MQA et la fenêtre couvrant toute la séquence retrouvent le marqueur. Les fenêtres courtes restent proches du hasard `1/8`. Augmenter la profondeur jusqu’à six couches rend le champ théorique supérieur à la distance, mais ne suffit pas à apprendre la tâche dans ce budget : la possibilité de transmettre une information ne garantit pas son apprentissage. La comparaison de profondeur repose sur une seule graine.

Le champ réceptif compte la position finale parmi les positions atteignables. Comparez donc `champ − 1` à la distance de 63 jetons. Avec six couches et `W = 16`, le champ théorique vaut 91, donc le marqueur est accessible par une chaîne de couches. Le score de cette passe renseigne sur l’optimisation de cette chaîne pendant 400 pas. La séparation du calcul de portée et de l’apprentissage permet de distinguer une impossibilité structurelle d’un échec dans le budget étudié.


## Exercice 3 — le point de croisement, mesure

**Objectif.** Trouver par la mesure la longueur `T` a partir de laquelle la bande explicite devient
plus rapide que le masque, pour une fenêtre `W = 64`.

**Indice.** Le point de croisement depend de `T/W` et de l'efficacite relative du `einsum` a cinq
dimensions contre le produit matriciel dense. Chercher la longueur ou les deux courbes se croisent
revient a chercher `T` tel que `T^2 ~ c * T * W`, avec `c` le surcout par élément de la bande.

**Étape 1.** Choisir au moins cinq longueurs croissantes, dont une sous `512` et une au-dessus de
`1024`.
**Étape 2.** Mesurer forward + backward des deux implementations pour chacune (fonction `Bench`).
**Étape 3.** Rapporter le rapport `masque / bande` et conclure sur l'ordre de grandeur du croisement.

**Pourquoi ca compte.** C'est la mesure qui separe « l'algorithme est asymptotiquement meilleur » de
« ce code-ci est plus rapide ». Un `O(T*W)` ecrit avec des opérations inefficaces peut rester plus
lent que le `O(T^2)` qu'il remplace sur toute la plage utile, et seul un chronometrage permet de le
savoir.

Pour reprendre le jeu proposé, utilisez `128, 256, 512, 1024, 1536` lorsque `lengths` est nul. Complétez `Croisement` et indiquez si les mesures établissent un croisement dans cette plage.

In [13]:
Dictionary<int, (double, double)> Croisement(int W = 64, int[] lengths = null)
{ // TODO : mesurer les deux modules ensemble avec Bench, pour chaque longueur
    return null;
}

var longueurs = new[] { 128, 256, 512, 1024, 1536 };
var measures = Croisement(64, longueurs);
if (measures is null)
    Console.WriteLine("Exercice 3 à compléter : résultat null");
else
    foreach (var(length, t)in measures)
        Console.WriteLine($"{length} masque {t.Item1:F2}, bande {t.Item2:F2}, gain {t.Item1 / t.Item2:F2}");


Exercice 3 à compléter : résultat null


## 7. Conclusion

Le cache dépend du nombre de têtes KV, tandis que la fenêtre limite les positions visibles. Les vérifications de calcul et gradients précèdent les mesures de temps et le test de rappel. Les résultats numériques et leurs limites sont ceux du replay ci-dessus.

[← TV-00a C# · RoPE](TV-00a-RoPE-from-scratch-CSharp.ipynb) | [TV-00d Python · Encodeurs →](TV-00d-Encodeurs-from-scratch-Python.ipynb) | [TV-04 Python · Recherche sur DiscoLoop →](TV-04-DiscoLoop-Python.ipynb)